In [ ]:
from dataclasses import dataclass
import json

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.nn.utils.parametrizations import weight_norm
from torch.utils.data import DataLoader, Dataset

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

## TCN Modules

In [ ]:
class OutputCrop1d(nn.Module):
    def __init__(self, crop_size: int):
        super().__init__()
        self.crop_size = crop_size

    def forward(self, x: torch.Tensor):
        return x[:, :, :-self.crop_size].contiguous()


class TemporalConvUnit(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        padding: int,
        dilation: int,
        stride: int = 1,
        dropout: float = 0.2,
        name: str | None = None
    ):
        super().__init__()
        self.name = name
        
        # Weight normalisation: https://arxiv.org/abs/1602.07868
        self.conv = weight_norm(
            nn.Conv1d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                padding=padding,
                dilation=dilation,
                stride=stride,
            )
        )
        self.conv.weight.data.normal_(0, 0.01)
        self.crop = OutputCrop1d(padding)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv, self.crop, self.relu, self.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)
    

class TemporalConvBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        padding: int,
        dilation: int,
        stride: int = 1,
        dropout: float = 0.2,
    ):
        """
        :param in_channels: Number of input channels.
            Corresponds to the number of features at each timestep in the input series.
        :param out_channels: Number of output channels.
            Corresponds to the number of features at each timestep in the output series.
        :param kernel_size: Number of weights per filter.
        :param padding: Size of padding to apply to both sides of the input
        """
        super().__init__()
        
        self.unit1 = TemporalConvUnit(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
            stride=stride,
            dropout=dropout,
        )
        self.unit2 = TemporalConvUnit(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
            stride=stride,
            dropout=dropout,
        )
        self.net = nn.Sequential(self.unit1, self.unit2)

        # Residual connection
        if in_channels != out_channels:
            self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=1)
            self.conv.weight.data.normal_(0, 0.01)
        else:
            self.conv = None
        
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.net(x)
        res = x if self.conv is None else self.conv(x)
        return self.relu(out + res)
    

class TemporalConvNetDecoder(nn.Module):
    def __init__(self, in_features: int, horizon: int = 1):
        super().__init__()
        """
        Linear decoder that maps the final hidden representation from the TCN 
        into the target forecasting horizon.

        :param in_features: Number of input features (channels) from the final TCN layer. 
            This corresponds to the number of learned feature maps at the last timestep.

        :param horizon: Number of future timesteps to predict. 
            The decoder outputs one value per step in the forecast horizon.
        """
        self.linear = nn.Linear(in_features=in_features, out_features=horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)


class TemporalConvNet(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: list[int],
        horizon: int, 
        kernel_size: int = 2,
        dropout: float = 0.2
    ):
        """
        :param in_features: Number of input features at each timestep in the time series.
            This corresponds to the number of input channels to the first convolutional layer.
        
        :param out_features: List specifying the number of output feature maps (channels) 
            for each temporal convolutional block in the network.
            For example, [16, 32, 64] creates three stacked convolutional blocks with
            16, 32, and 64 output channels, respectively.
        
        :param horizon: Number of future timesteps to predict i.e. the forecasting horizon
        
        :param kernel_size: Size of the temporal convolution kernel.
            Controls the receptive field of each convolutional layer.
        
        :param dropout: Dropout probability applied after each convolutional layer.
        """
        super().__init__()
        
        layers = []
        n_layers = len(out_features)
        for i in range(n_layers):
            in_channels = in_features if i == 0 else out_features[i - 1]
            out_channels = out_features[i]
            dilation_size = 2 ** i
            conv_block = TemporalConvBlock(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                dilation=dilation_size,
                padding=(kernel_size - 1) * dilation_size,
                dropout=dropout
            )
            layers.append(conv_block)
                

        self.encoder = nn.Sequential(*layers)
        self.decoder = TemporalConvNetDecoder(out_features[-1], horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        encoded = self.encoder(x)
        # Only select the final timestep feature maps
        # for forecasting
        return self.decoder(encoded[:, :, -1])

## UCI Dataset

In [ ]:
from datetime import datetime, timedelta

import polars as pl

In [ ]:
UCI_CLIENT_SITES_TO_VALIDATE: list[str] = [
    "MT_156",
    "MT_162",
    "MT_189",
    "MT_190",
    "MT_191",
    "MT_205",
    "MT_212",
    "MT_217",
    "MT_240",
    "MT_251",
    "MT_261",
    "MT_262",
    "MT_263",
    "MT_267",
    "MT_280",
    "MT_297",
    "MT_299",
    "MT_307",
    "MT_321",
    "MT_329",
]

In [ ]:
KAGGLE_INPUT_PATH = "/kaggle/input"
KAGGLE_OUTPUT_PATH = "/kaggle/working"
UCI_DATA_PATH = f"{KAGGLE_INPUT_PATH}/uci-electricity-load-2011-2014/preprocessed.pq"

CLIENT_SITE = "MT_156"
assert CLIENT_SITE in UCI_CLIENT_SITES_TO_VALIDATE

FEATURES = [
    "demand_scaled",
    "sin_hour_of_day",
    "cos_hour_of_day",
    "sin_day_of_week",
    "cos_day_of_week",
    "sin_month_of_year",
    "cos_month_of_year",
    "is_weekend"
]

FREQUENCY_MINUTES = 15
VALIDATION_START = datetime(2014, 12, 1)
VALIDATION_WINDOW = timedelta(days=2)
N_FOLDS = 10
INPUT_SEQ_LENGTH= int(30 * 24 * 60 / FREQUENCY_MINUTES)
OUTPUT_SEQ_LENGTH = int(VALIDATION_WINDOW.days * 24 * 60 / FREQUENCY_MINUTES)


In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, timeseries: np.ndarray, in_seq_length: int, out_seq_length: int, target_col_idx: int = 0):
        super().__init__()
        if timeseries.ndim < 2:
            timeseries = timeseries.reshape(-1, 1)
        elif timeseries.ndim > 2:
            raise ValueError("Expecting input array with at most two dimensions.")
        
        self.timeseries = torch.tensor(timeseries, dtype=torch.float32)
        self.in_seq_length = in_seq_length
        self.out_seq_length = out_seq_length
        self.target_col_index = target_col_idx

    def __len__(self):
        return len(self.timeseries) - self.in_seq_length - self.out_seq_length
    
    def __getitem__(self, idx) -> tuple[torch.Tensor, torch.Tensor]:
        x_start, x_end = int(idx), int(idx + self.in_seq_length)
        x = self.timeseries[x_start: x_end]
        
        y_start, y_end = int(x_end), int(x_end + self.out_seq_length)
        y = self.timeseries[y_start: y_end, [self.target_col_index]]
        return x, y

In [ ]:
def compute_features(df: pl.DataFrame, mean_train: float, std_train: float) -> pl.DataFrame:
    return (
        df
        .with_columns(
            demand_scaled=(pl.col("demand") - mean_train) / std_train,
            
            hour_of_day=pl.col("timestamp").dt.hour(),
            sin_hour_of_day=(pl.col("timestamp").dt.hour() * 2 * np.pi / 24).sin(),
            cos_hour_of_day=(pl.col("timestamp").dt.hour() * 2 * np.pi / 24).cos(),

            day_of_week=pl.col("timestamp").dt.weekday(),
            sin_day_of_week=(pl.col("timestamp").dt.weekday() * 2 * np.pi / 7).sin(),
            cos_day_of_week=(pl.col("timestamp").dt.weekday() * 2 * np.pi / 7).cos(),
            is_weekend=(pl.col("timestamp").dt.weekday() >= 6).cast(pl.Float64),
            
            month_of_year=pl.col("timestamp").dt.month(),
            sin_month_of_year=(pl.col("timestamp").dt.month() * 2 * np.pi / 12).sin(),
            cos_month_of_year=(pl.col("timestamp").dt.month() * 2 * np.pi / 12).cos(),
        )
    )


def train_tcn_model(model: TemporalConvNet, dataloader: DataLoader, n_epochs: int = 50):
    model = model.to(DEVICE)

    loss_fn = nn.MSELoss()
    optimizer = AdamW(model.parameters(), lr=1e-03)

    model.train()
    epoch_loss, batch_loss = [], []
    for epoch in range(n_epochs):
        for batch_X, batch_y in dataloader:
            
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            
            # Typically sequence modelling have (batch_size, in_seq_length, num_input_fetures)
            # But CNNs work with (batch_size, num_input_features, in_seq_length) convention
            batch_X = batch_X.permute(0, 2, 1)
            y_hat = model(batch_X)

            # Output dimension is (batch_size, out_seq_length)
            y_hat = y_hat.unsqueeze(-1)
            loss = loss_fn(batch_y, y_hat)
            loss.backward()
            optimizer.step()
            
            loss_detach = float(loss.detach())
            batch_loss.append(loss_detach)
    
        epoch_loss.append(loss_detach)

    metadata = {"batch_losses": batch_loss, "epoch_losses": epoch_loss}
    return model, metadata


def predict_tcn_model(model: TemporalConvNet, X_test: torch.Tensor) -> torch.Tensor:
    model.eval()
    with torch.no_grad():
        X_test = X_test.to(DEVICE)
        y_hat = model(X_test.permute(0, 2, 1))
        y_hat = y_hat.squeeze()
    return y_hat
    

In [ ]:
# Load data for client site

df = pl.read_parquet(UCI_DATA_PATH)
client_df = df.filter(pl.col("client") == CLIENT_SITE).sort(by="timestamp")

In [ ]:
# Initialise model
model = TemporalConvNet(
    in_features=len(FEATURES),
    out_features=[4, 8, 16, 32],
    horizon=OUTPUT_SEQ_LENGTH,
    kernel_size=2,
    dropout=0.2,
)

# Construct training dataset and dataloaders
train_df = (
    client_df
    .filter(pl.col("timestamp").lt(VALIDATION_START))
    .select(pl.col("timestamp"), pl.col("demand"))
    .sort(by="timestamp")
)
mean_train, std_train = train_df["demand"].mean(), train_df["demand"].std()
train_df = compute_features(train_df, mean_train, std_train)

train_ds = TimeSeriesDataset(
    timeseries=train_df[FEATURES].to_numpy(),
    in_seq_length=INPUT_SEQ_LENGTH,
    out_seq_length=OUTPUT_SEQ_LENGTH,
    target_col_idx=0,
)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

# Train model and save outputs
model, losses = train_tcn_model(model, train_dl, n_epochs=5)

# TODO: Save model

epoch_losses_path = f"{KAGGLE_OUTPUT_PATH}/epoch_train_loss_{CLIENT_SITE}.json"
with open(epoch_losses_path, "w") as fp:
    json.dump(losses["epoch_losses"], fp)

batch_losses_path = f"{KAGGLE_OUTPUT_PATH}/batch_train_loss_{CLIENT_SITE}.json"
with open(batch_losses_path, "w") as fp:
    json.dump(losses["batch_losses"], fp)

In [ ]:
# k-fold validation
for i in range(N_FOLDS):    
    val_start = VALIDATION_START + VALIDATION_WINDOW * i
    val_end = val_start + VALIDATION_WINDOW
    feature_df = (
        client_df
        .filter(pl.col("timestamp").lt(val_start))
        .select(pl.col("timestamp"), pl.col("demand"))
        .sort(by="timestamp")
        .slice(offset=-INPUT_SEQ_LENGTH)
    )
    target_df = (
        client_df
        .filter(pl.col("timestamp").is_between(val_start, val_end, closed="left"))
        .select(pl.col("timestamp"), pl.col("demand"))
        .sort(by="timestamp")
    )
    
    # Predict
    feature_df = compute_features(feature_df, mean_train, std_train)
    feature_array = feature_df[FEATURES].to_numpy()
    feature_ts = torch.from_numpy(feature_array).float().unsqueeze(dim=0)
    y_hat = predict_tcn_model(model, feature_ts)
    y_hat = y_hat.cpu().numpy() * std_train + mean_train

    # Save training losses + forecasts
    forecast_output_path = f"{KAGGLE_OUTPUT_PATH}/forecasts_{CLIENT_SITE}_fold_{i}.pq"
    target_df.with_columns(forecast=y_hat).to_pandas().to_parquet(forecast_output_path)